In [145]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
import numpy as np

splits = [
    'algebra', 'counting_and_probability', 'geometry',
    'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
]
final_dataset = {}
total_length = 3000
per_split = total_length // len(splits)
for split in splits:
    # Load and concatenate train/test
    ds = load_dataset("EleutherAI/hendrycks_math", split)
    full_ds = concatenate_datasets([ds['train'], ds['test']])
    full_ds = full_ds.shuffle(seed=42)
    
    # Get all levels present
    levels = [4, 5]
    per_level = per_split // len(levels)
    selected_indices = []
    
    for level in levels:
        # Find indices for this level
        level_indices = [i for i, l in enumerate(full_ds['level']) if l == 'Level ' + str(level)]
        np.random.seed(42)
        np.random.shuffle(level_indices)
        # Take up to per_level from this level
        add_remaining = 4 if level == 5 and split == splits[0] else 0
        selected_indices.extend(level_indices[:per_level + add_remaining])
    # If not enough, fill up to 1000 with random remaining
    if len(selected_indices) < per_split:
        remaining = list(set(range(len(full_ds))) - set(selected_indices))
        np.random.seed(42)
        np.random.shuffle(remaining)
        selected_indices.extend(remaining[:per_split - len(selected_indices)])
    
    # If fewer than 1000 total, just use all
    if len(full_ds) < per_split:
        selected_indices = list(range(len(full_ds)))
    
    # Select and shuffle final
    np.random.seed(42)
    np.random.shuffle(selected_indices)
    final_dataset[split] = full_ds.select(selected_indices)
for spl, data in final_dataset.items():
    final_dataset[spl] = data.add_column('split',[spl] * len(data))
df = concatenate_datasets(list(final_dataset.values())).shuffle(42)

# Example: final_dataset['algebra'] is your processed dataset for that split

Flattening the indices:   0%|          | 0/432 [00:00<?, ? examples/s]

In [163]:
import glob
import json
fnames = glob.glob(f"responses/math/math_train_*.json")
for fname in fnames:
    file = json.load(open(fname,"r"))
    file_vals = list(file.values())
    list_format = any([isinstance(fv, list) for fv in file_vals])
    if list_format:
        file = {
                k: v[0] 
                if isinstance(v, list) and len(v) == 1
                else v
                for k, v in file.items()
                }
        json.dump(file, open(fname, "w"), indent=2)


In [156]:
solutions = dict(zip(range(0, len(df['solution'])),  df['solution']))
questions = dict(zip(range(0, len(df['problem'])),  df['problem']))

In [ ]:
with open("sources/math_train_sources.json", "w") as f:
    import json
    json.dump(questions, f, indent=2)
with open("responses/math/math_train_human_responses_merged.json", "w") as f:
    import json
    json.dump(solutions, f, indent=2)


In [116]:
with open("responses/apps/apps_train_human_responses_merged.json", "r") as f:
    apple = json.load(f)
with open("sources/apps_train_sources.json","w") as fp:
    json.dump(apple, fp, indent=2)

### ArXiv Summarization

In [185]:
arxiv_test = open("arxiv-dataset/train.txt","r").readlines()
import numpy as np
np.random.seed(42)
shuffled_indices = np.random.permutation(len(arxiv_test))
print(len(shuffled_indices))

203037


In [198]:
arxiv_subset_metadata = [arxiv_test[i] for i in shuffled_indices[:5000]]
with open("sources/arxiv_train_metadata.jsonl","w") as f:
    for line in arxiv_subset_metadata:
        f.write(line)


In [1]:
with open("sources/arxiv_train_metadata.jsonl","r") as f:
    open_arxiv = f.readlines()

In [4]:
import json
open_arxiv = [json.loads(line) for line in open_arxiv]

In [35]:
import tiktoken
from transformers import AutoTokenizer
gpt_tok = tiktoken.encoding_for_model("gpt-3.5-turbo")
gpt_limit = 16000
hf_tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", use_fast=True)
hf_limit = 13100
id_to_text = {oa['article_id'] :
                "\n\n".join([ # bridging sections, indexed by article id
                    "\n".join(["## " + label] + section) # uniting sentence headers with sections, adding by sentence
                    for label, section in zip(oa['section_names'], oa['sections']) # zip iterator
                ])[:
                   ]
            for oa in open_arxiv} #for each entry

for id, text in id_to_text.items():
    gpt_tokens = gpt_tok.encode(text); hf_tokens = hf_tok.encode(text)
    gpt_truncated = text; hf_truncated = text
    if len(gpt_tokens) > gpt_limit:
        gpt_truncated = gpt_tok.decode(gpt_tokens[:gpt_limit])
    if len(hf_tokens) > hf_limit:
        hf_truncated = hf_tok.decode(hf_tokens[:hf_limit])
    id_to_text[id] = min(gpt_truncated, hf_truncated, key=len)

id_to_abstract = {oa['article_id']: oa['abstract_text'] for oa in open_arxiv}

with open("sources/arxiv_train_sources.json", "w") as f:
    json.dump(id_to_text, f, indent=2)
import os
os.makedirs("responses/arxiv",exist_ok=True)
with open("responses/arxiv/arxiv_train_human_responses_merged.json","w") as f:
    json.dump(id_to_abstract, f, indent=2)

Token indices sequence length is longer than the specified maximum sequence length for this model (143181 > 131072). Running this sequence through the model will result in indexing errors


In [ ]:
from martian_apart_hack_sdk import martian_client

mc = martian_client.MartianClient(
    api_url = "http://withmartian.com/api",
    api_key = os.getenv("MARTIAN_API_KEY")
)
mc.org_id


'75b95d14-2465-48e6-8975-a2d0a82b4d67'